# Parcial 3 - Reconocimiento de prendas con CNN (Fashion MNIST)

Este notebook:
1. Carga Fashion MNIST (Kaggle o Keras).
2. Define y entrena **3 arquitecturas CNN** con el mismo protocolo.
3. Compara metricas (accuracy, precision, recall, F1, matriz de confusion, parametros, tiempos).
4. Selecciona el mejor modelo y lo guarda como `modelos/mejor_modelo.keras` para la aplicacion con camara.

**Integrantes:** [Nombre 1], [Nombre 2], [Nombre 3]

> En Google Colab: Entorno de ejecucion > Cambiar tipo > GPU (T4). Luego descargue la carpeta `modelos/` y `resultados/`.

In [ ]:
import os, time, json, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

os.makedirs("modelos", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Carga de datos

Si descargo los CSV de Kaggle (`fashion-mnist_train.csv` y `fashion-mnist_test.csv`), indique la carpeta en `RUTA_KAGGLE`.
Si la deja vacia, se usa la copia oficial del mismo dataset incluida en Keras.

In [ ]:
RUTA_KAGGLE = ""  # ejemplo: "/content/fashionmnist" o "data"

CLASES_EN = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
             "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CLASES_ES = ["Camiseta", "Pantalon", "Sueter", "Vestido", "Abrigo",
             "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botin"]


def cargar_datos():
    train_csv = os.path.join(RUTA_KAGGLE, "fashion-mnist_train.csv")
    test_csv = os.path.join(RUTA_KAGGLE, "fashion-mnist_test.csv")
    if RUTA_KAGGLE and os.path.exists(train_csv) and os.path.exists(test_csv):
        print("Cargando CSV de Kaggle...")
        tr = pd.read_csv(train_csv)
        te = pd.read_csv(test_csv)
        x_tr = tr.drop(columns="label").values.reshape(-1, 28, 28).astype("uint8")
        y_tr = tr["label"].values.astype("int64")
        x_te = te.drop(columns="label").values.reshape(-1, 28, 28).astype("uint8")
        y_te = te["label"].values.astype("int64")
    else:
        print("Cargando Fashion MNIST desde Keras...")
        (x_tr, y_tr), (x_te, y_te) = keras.datasets.fashion_mnist.load_data()
    return x_tr, y_tr, x_te, y_te


x_train_full, y_train_full, x_test, y_test = cargar_datos()

# Agregar canal -> (N, 28, 28, 1). La normalizacion (1/255) va DENTRO del modelo.
x_train_full = x_train_full[..., None].astype("float32")
x_test = x_test[..., None].astype("float32")

# Validacion estratificada (10 %)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.10, stratify=y_train_full, random_state=SEED)

print("Train:", x_train.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)
print("Distribucion train:", np.bincount(y_train))

## 2. Exploracion: una imagen por clase

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train[idx].squeeze(), cmap="gray")
    ax.set_title(f"{i}: {CLASES_ES[i]}")
    ax.axis("off")
plt.suptitle("Fashion MNIST - una muestra por clase")
plt.tight_layout()
plt.savefig("resultados/muestras_dataset.png", dpi=150)
plt.show()

## 3. Arquitecturas

Todas comparten la misma entrada: normalizacion (`Rescaling`) y aumento de datos leve
(solo activo durante el entrenamiento).

| # | Modelo | Idea principal |
|---|---|---|
| 1 | CNN Simple | Linea base: 2 bloques conv + densa |
| 2 | CNN Profunda BN | Estilo VGG con BatchNorm y Dropout |
| 3 | Mini-ResNet | Bloques residuales + Global Average Pooling |

In [ ]:
def entrada_comun():
    """Entrada + normalizacion + aumento de datos (solo en entrenamiento)."""
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomTranslation(0.08, 0.08)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomRotation(0.05)(x)
    return inputs, x


def modelo_cnn_simple():
    inputs, x = entrada_comun()
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Simple")


def modelo_cnn_profunda_bn():
    inputs, x = entrada_comun()
    for filtros, drop in [(32, 0.25), (64, 0.25), (128, 0.25)]:
        for _ in range(2):
            x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Profunda_BN")


def bloque_residual(x, filtros, stride=1):
    atajo = x
    y = layers.Conv2D(filtros, 3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if stride != 1 or atajo.shape[-1] != filtros:
        atajo = layers.Conv2D(filtros, 1, strides=stride, padding="same", use_bias=False)(atajo)
        atajo = layers.BatchNormalization()(atajo)
    y = layers.Add()([y, atajo])
    return layers.ReLU()(y)


def modelo_mini_resnet():
    inputs, x = entrada_comun()
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = bloque_residual(x, 32)
    x = bloque_residual(x, 64, stride=2)
    x = bloque_residual(x, 64)
    x = bloque_residual(x, 128, stride=2)
    x = bloque_residual(x, 128)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="Mini_ResNet")


CONSTRUCTORES = {
    "CNN_Simple": modelo_cnn_simple,
    "CNN_Profunda_BN": modelo_cnn_profunda_bn,
    "Mini_ResNet": modelo_mini_resnet,
}

for nombre, fn in CONSTRUCTORES.items():
    m = fn()
    print(f"{nombre}: {m.count_params():,} parametros")

modelo_cnn_simple().summary()

## 4. Entrenamiento (mismo protocolo para los tres modelos)

In [ ]:
EPOCAS = 30
BATCH = 128


def entrenar(nombre, constructor):
    print("\n" + "=" * 70)
    print("Entrenando:", nombre)
    print("=" * 70)
    keras.backend.clear_session()
    tf.random.set_seed(SEED)
    modelo = constructor()
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    ]
    t0 = time.time()
    hist = modelo.fit(x_train, y_train, validation_data=(x_val, y_val),
                      epochs=EPOCAS, batch_size=BATCH, callbacks=callbacks, verbose=2)
    t_entrenamiento = time.time() - t0
    modelo.save(f"modelos/{nombre}.keras")
    return modelo, hist.history, t_entrenamiento


modelos, historiales, tiempos = {}, {}, {}
for nombre, fn in CONSTRUCTORES.items():
    modelos[nombre], historiales[nombre], tiempos[nombre] = entrenar(nombre, fn)

## 5. Curvas de aprendizaje

In [ ]:
for nombre, h in historiales.items():
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["loss"], label="entrenamiento")
    ax[0].plot(h["val_loss"], label="validacion")
    ax[0].set_title(f"{nombre} - Perdida"); ax[0].set_xlabel("epoca"); ax[0].legend()
    ax[1].plot(h["accuracy"], label="entrenamiento")
    ax[1].plot(h["val_accuracy"], label="validacion")
    ax[1].set_title(f"{nombre} - Accuracy"); ax[1].set_xlabel("epoca"); ax[1].legend()
    plt.tight_layout()
    plt.savefig(f"resultados/curvas_{nombre}.png", dpi=150)
    plt.show()

## 6. Evaluacion en el conjunto de prueba

In [ ]:
def evaluar(nombre, modelo):
    probs = modelo.predict(x_test, batch_size=256, verbose=0)
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)

    # Tiempo de inferencia por imagen (1000 imagenes, tras calentamiento)
    modelo.predict(x_test[:64], verbose=0)
    t0 = time.time()
    modelo.predict(x_test[:1000], batch_size=1000, verbose=0)
    ms_img = (time.time() - t0) / 1000 * 1000

    return {
        "Modelo": nombre,
        "Parametros": modelo.count_params(),
        "Accuracy": acc,
        "Precision_macro": prec,
        "Recall_macro": rec,
        "F1_macro": f1,
        "Epocas_entrenadas": len(historiales[nombre]["loss"]),
        "Tiempo_entrenamiento_s": tiempos[nombre],
        "Inferencia_ms_por_imagen": ms_img,
    }, y_pred


resultados, predicciones = [], {}
for nombre, modelo in modelos.items():
    fila, y_pred = evaluar(nombre, modelo)
    resultados.append(fila)
    predicciones[nombre] = y_pred

df = pd.DataFrame(resultados).sort_values("F1_macro", ascending=False).reset_index(drop=True)
df.to_csv("resultados/comparacion_modelos.csv", index=False)
display(df.round(4))

In [ ]:
# Grafica comparativa
metricas = ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]
ax = df.set_index("Modelo")[metricas].plot(kind="bar", figsize=(10, 5), ylim=(df[metricas].min().min() - 0.02, 1.0))
ax.set_title("Comparacion de modelos (conjunto de prueba)")
ax.set_ylabel("Valor"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("resultados/comparacion_metricas.png", dpi=150)
plt.show()

## 7. Matriz de confusion y reporte por clase (todos los modelos)

In [ ]:
def graficar_confusion(nombre, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CLASES_ES, rotation=45, ha="right")
    ax.set_yticklabels(CLASES_ES)
    umbral = cm.max() / 2
    for i in range(10):
        for j in range(10):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > umbral else "black")
    ax.set_xlabel("Prediccion"); ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusion - {nombre}")
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(f"resultados/matriz_confusion_{nombre}.png", dpi=150)
    plt.show()


for nombre, y_pred in predicciones.items():
    graficar_confusion(nombre, y_pred)
    reporte = classification_report(y_test, y_pred, target_names=CLASES_ES, digits=4)
    with open(f"resultados/reporte_{nombre}.txt", "w", encoding="utf-8") as f:
        f.write(reporte)
    print(f"\n--- {nombre} ---\n{reporte}")

## 8. Seleccion del mejor modelo

Criterio: mayor **F1 macro** en prueba (desempate por accuracy y menor numero de parametros).

In [ ]:
df_sel = df.sort_values(["F1_macro", "Accuracy", "Parametros"], ascending=[False, False, True]).reset_index(drop=True)
mejor = df_sel.loc[0, "Modelo"]
print("MEJOR MODELO:", mejor)
print(df_sel.loc[0].to_string())

shutil.copy(f"modelos/{mejor}.keras", "modelos/mejor_modelo.keras")
with open("modelos/clases.json", "w", encoding="utf-8") as f:
    json.dump({"en": CLASES_EN, "es": CLASES_ES, "mejor_modelo": mejor}, f, ensure_ascii=False, indent=2)
print("Guardado: modelos/mejor_modelo.keras")

## 9. Errores del mejor modelo

In [ ]:
y_pred_mejor = predicciones[mejor]
idx_err = np.where(y_pred_mejor != y_test)[0]
print(f"Errores: {len(idx_err)} de {len(y_test)} ({len(idx_err) / len(y_test):.2%})")

probs_mejor = modelos[mejor].predict(x_test, batch_size=256, verbose=0)
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax, i in zip(axes.ravel(), np.random.RandomState(SEED).choice(idx_err, 18, replace=False)):
    ax.imshow(x_test[i].squeeze(), cmap="gray"); ax.axis("off")
    ax.set_title(f"Real: {CLASES_ES[y_test[i]]}\nPred: {CLASES_ES[y_pred_mejor[i]]} "
                 f"({probs_mejor[i].max():.0%})", fontsize=8, color="red")
plt.suptitle(f"Ejemplos mal clasificados - {mejor}")
plt.tight_layout()
plt.savefig("resultados/errores_mejor_modelo.png", dpi=150)
plt.show()

## 10. Siguientes pasos

1. Descargue `modelos/mejor_modelo.keras`.
2. Ejecute la aplicacion con camara: `python app_camara.py --modelo modelos/mejor_modelo.keras`.
3. Use `resultados/` para el informe tecnico.